In [8]:
import sys
sys.path.append("/home/yigit/codebase/dna2vec/evaluate")
sys.path.append("/home/yigit/codebase/dna2vec/src")
import os
import hydra
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from itertools import product
from omegaconf import DictConfig
from tqdm import tqdm
from helpers import initialize_pinecone, align_real_reads, query_and_align, calculate_SW_and_Cosine_similarity
from dna2vec.simulate import real_mapped_reads #simulate_mapped_reads
import matplotlib.pyplot as plt
from typing import List, Optional, Union   
from pysam.libcalignedsegment import AlignedSegment
from dataclasses import dataclass
from dna2vec.utils import (
    download_human_reference_genome,
    get_cache_dir,
    load_human_reference_genome,
)
from collections import defaultdict
from typing import Tuple
import pysam


In [9]:
@dataclass
class ReadAndReference:
    """
    Class for storing a read and its reference.

    Attributes:
        read: The read.
        reference: The reference sequence as a string.
    """

    read: AlignedSegment
    reference: Union[str, None] = None
    id: Union[str, None] = None

    def is_mapped(self) -> bool:
        return self.reference is not None

    def get_pair_as_string(self) -> Tuple[str, Union[str, None]]:
        """
        Get the pair as a tuple of strings.
        """
        return (self.read.query_sequence, self.reference)

In [21]:
def load_real_reads_from_disk(
    bam_file: Path,
) -> list[AlignedSegment]:
    """
    Read in the simulated reads

    Args:
        simulated_path: Path to the simulated reads.

    Returns:
        List of aligned segments.
    """
    logging.info(f"Reading real reads from {bam_file}")
    print(bam_file)

    # load using pysam
    aligned_segments = pysam.AlignmentFile(str(bam_file), "rb")
    return aligned_segments

def map_real_reads_to_reference(
    reads: List[AlignedSegment],
    reference: Optional[Path] = None,
) -> List[ReadAndReference]:
    """
    Map a read to a reference genome.
    """
    reference = load_human_reference_genome(reference)

    id2read = defaultdict(list)
    unmapped_reads = []
    # for read in reads.fetch("chr2",1018500,2020000):
    #     _id = "chr2"

    #     if len(unmapped_reads) > 2000:
    #         break
    #     unmapped_read = ReadAndReference(read=read)
    #     id2read[_id].append(unmapped_read)
    #     unmapped_reads.append(unmapped_read)

    # === Step 1: Load chr2 deletions from VCF ===
    deletions = []
    vcf_path = "/home/yigit/codebase/dna2vec/human_deletions_GS_correct.vcf"
    with open(vcf_path, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.strip().split("\t")
            chrom = parts[0]
            if chrom != "2" and chrom != "chr2":  # Accept both notations
                continue
            start = int(parts[1])
            info = dict(x.split("=") for x in parts[7].split(";") if "=" in x)
            end = int(info["END"])
            svlen = int(info["SVLEN"])
            if svlen > 50:
                deletions.append((chrom, start, end))

    print(f"Loaded {len(deletions)} deletions on chr2.")
    for chrom, start, end in tqdm(deletions, desc="Processing chr2 deletions"):
        try:
            for read in tqdm(reads.fetch(chrom, start-249, end+249), desc="Processing reads"):
                if read.cigartuples:
                    has_deletion = any(op == 2 and length >= 1 for op, length in read.cigartuples)  # D
                    has_soft_clip = any(op == 4 and length >= 1 for op, length in read.cigartuples)  # S
                    has_skipped_region = any(op == 3 and length >= 1 for op, length in read.cigartuples)  # N
                    
                    if has_deletion or has_soft_clip or has_skipped_region:
                        _id = chrom
                        unmapped_read = ReadAndReference(read=read)
                        id2read[_id].append(unmapped_read)
                        unmapped_reads.append(unmapped_read)

                        if len(unmapped_reads) >= 200:
                            break
            if len(unmapped_reads) >= 200:
                break
        except ValueError:
            print(f"Region doesn't exist in BAM: {chrom}:{start}-{end}")
            continue
        
    seq_offset = 0
    for seq in reference:
        matches = id2read[seq.id]
        for match in matches:
            read = match.read
            start = read.reference_start
            end = read.reference_end
            length = read.query_length
            original_sequence = seq.seq[start : start + length]
            read_sequence = seq.seq[start : end]
            match.reference = str(original_sequence)
            match.read_sequence = str(read_sequence)
            match.id = seq.id
            match.seq_offset = seq_offset
            assert read.query_sequence == read.seq
        seq_offset += len(seq.seq)
    return unmapped_reads

def real_mapped_reads(
    bam_file: Union[Path, None] = None,
    reference_genome: Union[Path, None] = None,
):
    """
    Simulates reads and maps them to the reference genome.
    """

    mapped_reads = map_real_reads_to_reference(
        reads=load_real_reads_from_disk(bam_file),
        reference=reference_genome,
    )

    return mapped_reads

In [23]:
bam_file = Path("/mnt/SSD1/yigit/dna_data/rde2/HG002.hs37d5.2x250.bam")
fasta_file_path = Path("/mnt/SSD1/yigit/dna_data/chromosome_2/hs37d5.fasta")
mapped_reads = real_mapped_reads(
    bam_file=bam_file,
    reference_genome=fasta_file_path,
)

/mnt/SSD1/yigit/dna_data/rde2/HG002.hs37d5.2x250.bam
/mnt/SSD1/yigit/dna_data/chromosome_2/hs37d5.fasta
Loaded 469 deletions on chr2.


Processing reads: 241it [00:00, 34289.74it/s]/469 [00:00<?, ?it/s]
Processing reads: 168it [00:00, 31345.33it/s]
Processing reads: 123it [00:00, 5666.86it/s]
Processing reads: 176it [00:00, 10778.49it/s]
Processing reads: 134it [00:00, 66885.25it/s]
Processing chr2 deletions:   1%|          | 4/469 [00:00<00:07, 62.18it/s]


In [30]:
x = mapped_reads[71]

In [31]:
x

ReadAndReference(read=<AlignedSegment('D00360:94:H2YT5BCXX:2:1216:19558:61752', flags=147=0x93, ref='2', zpos=1771728, mapq=70, cigar='106S144M', ...)>, reference='GGTGGGGGAGGTGACTTTTTTTTTTTTTTTTTTACAGGTAAGAGGAGAAACAATTTGGTATCAAAAGAGCGTCCTGACTGGCCTCCCTGCCTCCAGCATCCCATCCAATCCACGGCCACTCACTCCAGAAGCCCGTCCAATCCGCAACCACTCACTCTTTCCAAGGCCAGCGCTCCGATGATGCTGTAGCCTCAGTCCCGGCCTCTGTCAAAGGCCCTCCAGCCCCTGACGCCAGGCCACCTTTGAGCTC', id='2')

In [35]:
# Assuming x.read is an instance of AlignedSegment
aligned_segment = x.read

# Print all attribute names and their values
for attr in dir(aligned_segment):
    # Skip private attributes and methods
    if not attr.startswith('_'):
        value = getattr(aligned_segment, attr)
        print(f"{attr}: {value}")

aend: 1771872
alen: 144
aligned_pairs: [(0, None), (1, None), (2, None), (3, None), (4, None), (5, None), (6, None), (7, None), (8, None), (9, None), (10, None), (11, None), (12, None), (13, None), (14, None), (15, None), (16, None), (17, None), (18, None), (19, None), (20, None), (21, None), (22, None), (23, None), (24, None), (25, None), (26, None), (27, None), (28, None), (29, None), (30, None), (31, None), (32, None), (33, None), (34, None), (35, None), (36, None), (37, None), (38, None), (39, None), (40, None), (41, None), (42, None), (43, None), (44, None), (45, None), (46, None), (47, None), (48, None), (49, None), (50, None), (51, None), (52, None), (53, None), (54, None), (55, None), (56, None), (57, None), (58, None), (59, None), (60, None), (61, None), (62, None), (63, None), (64, None), (65, None), (66, None), (67, None), (68, None), (69, None), (70, None), (71, None), (72, None), (73, None), (74, None), (75, None), (76, None), (77, None), (78, None), (79, None), (80, None)

In [42]:
print(x.read_sequence)
print(x.read.query_sequence)
print(len(x.read.query_sequence))
print(x.read.cigarstring)


GGTGGGGGAGGTGACTTTTTTTTTTTTTTTTTTACAGGTAAGAGGAGAAACAATTTGGTATCAAAAGAGCGTCCTGACTGGCCTCCCTGCCTCCAGCATCCCATCCAATCCACGGCCACTCACTCCAGAAGCCCGTCCAATCCG
CCCCACACTCCTTGTCTCTCCTCCTCCGTCAGTCTCCTCCCTACCTCTTCCCCCCCCCCTCTCTCTACAACATCTCCTCCTTAGTTTTTGTGTGGTGGGTGTGGGGGGTGGGGGAGGTGTTTTTTTTTTTTTTTTTTTTACAGGTAAGAGGAGAAACAATTTGGTATCAAAAGAGCGTCCTGACTGGCCTCCCTGCCTCCAGCATCCCATCCAATCCACGGCCACTCACTCCAGAAGCCCGTCCAATCCG
250
106S144M


In [38]:
def find_differences(str1, str2):
    differences = []
    # Determine the length of the shorter string to avoid index errors
    min_length = min(len(str1), len(str2))
    
    # Compare characters index by index
    for i in range(min_length):
        if str1[i] != str2[i]:
            differences.append((i, str1[i], str2[i]))
    
    # Check for any remaining characters in the longer string
    if len(str1) > min_length:
        for i in range(min_length, len(str1)):
            differences.append((i, str1[i], None))
    elif len(str2) > min_length:
        for i in range(min_length, len(str2)):
            differences.append((i, None, str2[i]))
    
    return differences

find_differences(x.read_sequence, x.read.query)

[(13, 'A', 'T'), (14, 'C', 'T')]